# MelanCare: class imbalance comparison

Naevi make up two-thirds of HAM10000. The baseline model corrects for this with a class-weighted loss.
This notebook trains the same model two more ways, so the three can be compared:

- **none**: no correction
- **sampler**: rare classes are oversampled during training

Everything else (data split, model, augmentations, epochs, seed) is identical to the baseline.
Takes about 60 minutes.

1. *Runtime → Change runtime type → **T4 GPU** → Save*
2. *Runtime → **Run all***
3. Keep this tab open. At the end, `imbalance_results.zip` downloads to your computer.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: choose Runtime → Change runtime type → T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Get the code

In [ ]:
import os
os.chdir("/content")
!rm -rf melancare && git clone -q https://github.com/i-oladele/melancare.git
os.chdir("/content/melancare")
!pip install -q -e .

## 2. Download the data (about 3.2 GB)

In [ ]:
%%bash
mkdir -p data/raw && cd data/raw
B=https://dataverse.harvard.edu/api/access/datafile
S=https://isic-archive.s3.amazonaws.com/challenges/2018
curl -sSL --retry 5 -o HAM10000_metadata.csv "$B/4338392?format=original"
curl -sSL --retry 5 -o ISIC2018_Task3_Test_GroundTruth.csv "$B/6924466?format=original"
curl -sSL --retry 5 -o HAM10000_segmentations.zip "$B/3838943"
curl -sSL --retry 5 -o ISIC2018_Task3_Training_Input.zip "$S/ISIC2018_Task3_Training_Input.zip"
curl -sSL --retry 5 -o ISIC2018_Task3_Test_Input.zip "$S/ISIC2018_Task3_Test_Input.zip"
ls -lh

## 3. Resize images and create lesion-level splits (same split as the baseline)

In [ ]:
!python scripts/prepare_data.py

## 4. Train without any imbalance correction

In [ ]:
!python scripts/train.py --run-name effb0_none --imbalance none --num-workers 2

## 5. Train with oversampling of rare classes

In [ ]:
!python scripts/train.py --run-name effb0_sampler --imbalance sampler --num-workers 2

## 6. Evaluate both

In [ ]:
!python scripts/evaluate.py --run runs/effb0_none --num-workers 2
!python scripts/evaluate.py --run runs/effb0_sampler --num-workers 2

In [ ]:
!python scripts/compare_runs.py runs/effb0_none runs/effb0_sampler --out runs/comparison.md
from IPython.display import Markdown, display
display(Markdown(open("runs/comparison.md").read()))

## Save the results
Colab deletes everything when the session ends. If the download doesn't start, open the **Files**
panel (folder icon on the left), find `imbalance_results.zip` in `melancare`, and choose **⋮ → Download**.

In [ ]:
!cd runs && zip -qr ../imbalance_results.zip effb0_none effb0_sampler
from google.colab import files
files.download("imbalance_results.zip")